# 2장 — Working with text dataRaschka, *Build a LLM (From Scratch)* 2.2 ~ 2.4절.세션 02에서 루트에 흩어져 있던 `verdict.py` / `tokenizer.py`를 이 노트북으로 합쳤다.남길 코드는 `src/tokenizer.py`에 있다.**코퍼스:** `data/the-verdict.txt`는 세션 02에서 손상본(줄 끝 백슬래시 82개, 토큰 4,772개)으로 드러나 세션 03(2026-09-05)에 Raschka 저장소의 정본으로 교체했다. 지금은 책과 같은 4,690개가 나온다.자세한 내용은 `docs/LLM_project_session_02.md` §2-A, `docs/LLM_project_session_03.md` §1.

## 2.2 텍스트를 토큰으로 나누기

In [1]:
import re
import sys

sys.path.append("..")

CORPUS = "../experiments/ch02/embedding_exec/annabel_lee.txt"

with open(CORPUS, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 1570
Annabel Lee
By Edgar Allan Poe

It was many and many a year ago,
   In a kingdom by the sea,
That a


In [2]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("Number of tokens:", len(preprocessed))
print(preprocessed[:30])

Number of tokens: 336
['Annabel', 'Lee', 'By', 'Edgar', 'Allan', 'Poe', 'It', 'was', 'many', 'and', 'many', 'a', 'year', 'ago', ',', 'In', 'a', 'kingdom', 'by', 'the', 'sea', ',', 'That', 'a', 'maiden', 'there', 'lived', 'whom', 'you', 'may']


## 2.3 토큰을 토큰 ID로 변환

In [3]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

Vocabulary size: 145
('!', 0)
('(', 1)
(')', 2)
(',', 3)
('.', 4)
(';', 5)
('A', 6)
('Allan', 7)
('And', 8)
('Annabel', 9)
('But', 10)
('By', 11)
('Can', 12)
('Chilling', 13)
('Coveted', 14)
('Edgar', 15)
('For', 16)
('Heaven', 17)
('I', 18)
('In', 19)
('It', 20)
('Lee', 21)
('Lee—', 22)
('My', 23)
('Nor', 24)
('Of', 25)
('Poe', 26)
('So', 27)
('Than', 28)
('That', 29)
('The', 30)
('To', 31)
('Went', 32)
('With', 33)
('Yes', 34)
('a', 35)
('above', 36)
('ago', 37)
('all', 38)
('and', 39)
('angels', 40)
('as', 41)
('away', 42)
('be', 43)
('beams', 44)
('beautiful', 45)
('blew', 46)
('bore', 47)
('bride', 48)
('bright', 49)
('bringing', 50)


### SimpleTokenizerV1책을 따라 직접 타이핑한 원본. 정리된 판은 `src/tokenizer.py`에 있다.

In [4]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [5]:
tokenizer = SimpleTokenizerV1(vocab)

text = "In a kingdom by the sea, I loved Annabel Lee."
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[19, 35, 75, 52, 122, 108, 3, 18, 83, 9, 21, 4]
In a kingdom by the sea, I loved Annabel Lee.


## 2.4 특수 문맥 토큰 — 여기서 V1이 무너진다아래 셀은 **의도적으로 실패한다.** 어휘는 *Annabel Lee*에서 만들었는데테스트 문장은 *the-verdict*에서 가져왔다. 어휘에 없는 `"`에서 `KeyError`가 난다.이것이 책이 `<|unk|>`와 `<|endoftext|>`를 도입하는 이유다.V1은 "모르는 단어"라는 개념 자체가 없다 — 어휘 밖의 것을 만나면 표현할 방법이 없어 죽는다.**다음 작업:** `SimpleTokenizerV2`를 만들어 이 셀이 통과하게 한다.

In [6]:
text = '''"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride.'''

try:
    print(tokenizer.encode(text))
except KeyError as e:
    print("KeyError:", e, "  <- 어휘에 없는 토큰")

KeyError: '"'   <- 어휘에 없는 토큰


## 모듈에서 불러 쓰기`src/tokenizer.py`로 옮긴 판. 정규식이 세 군데 중복되어 있던 것을`preprocess()` 하나로 합쳤다.

In [7]:
from src.tokenizer import SimpleTokenizerV1 as Tokenizer, build_vocab, preprocess

vocab2 = build_vocab(raw_text)
tok = Tokenizer(vocab2)

text = "In a kingdom by the sea, I loved Annabel Lee."
assert tok.decode(tok.encode(text)) == text
print("왕복 성공:", tok.decode(tok.encode(text)))

왕복 성공: In a kingdom by the sea, I loved Annabel Lee.


## 관찰 — 구분자가 모르는 문자책의 정규식은 `--`(하이픈 두 개)는 알지만 `—`(em dash, U+2014)는 모른다.그래서 `love—`가 통째로 하나의 토큰이 된다.모델 입장에서 `love`와 `love—`는 **아무 관계도 없는 두 단어**다.

In [8]:
odd = [w for w in sorted(vocab2) if any(ord(c) > 127 for c in w)]
print(odd)
print("\n'love' in vocab:", "love" in vocab2)
print("'love—' in vocab:", "love\u2014" in vocab2)

['Lee—', 'darling—my', 'love—', 'me—', 'sea—', 'we—', 'wingéd', '—that']

'love' in vocab: True
'love—' in vocab: True


### 직접 해 볼 것- `tiktoken`으로 `love—`, `wingèd`를 인코딩해서 몇 조각으로 쪼개지는지 확인 (2.5절 BPE)- 같은 문장의 한국어판/영어판 토큰 수 비교 — 세션 01의 "2~3배" 주장을 실측으로 검증- 정규식에서 `--`를 빼고 돌려서 어휘가 어떻게 변하는지 관찰

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids =[]
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
            

In [10]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [11]:
import tiktoken
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
